In [3]:
import pandas as pd
import numpy as np

In [56]:
class Option:
    def __init__(self, strike: float, premium: float, expiry: str):
        self.strike = strike
        self.premium = premium
        self.expiry = expiry
        self.spot_price = None

    def insert_spot(self, spot_price: float):
        self.spot_price = spot_price
    
    def short_call(self):
        # Payoff for short call: premium received - intrinsic value of the option
        intrinsic_value = np.maximum(0, self.spot_price - self.strike)
        return self.premium - intrinsic_value
    
    def long_call(self):
        # Payoff for long call: intrinsic value - premium paid
        intrinsic_value = np.maximum(0, self.spot_price - self.strike)
        return np.maximum(intrinsic_value - self.premium, -self.premium)
    
    def short_put(self):
        # Payoff for short put: premium received - intrinsic value of the option
        intrinsic_value = np.maximum(0, self.strike - self.spot_price)
        return self.premium - intrinsic_value
    
    def long_put(self):
        # Payoff for long put: intrinsic value - premium paid
        intrinsic_value = np.maximum(0, self.strike - self.spot_price)
        return np.maximum(intrinsic_value - self.premium, -self.premium)


class IronCondor:
    def __init__(self, direction: str, strikes: dict, premiums: dict, expiry: str):
        """
        Initialize the Iron Condor strategy with strikes and premiums for the options.
        
        direction: long, (short) Iron Condor
        1. Short (Long) 1 OTM Put (strike 1)
        2. Long (Short) 1 close to ATM Put (strike 2)
        3. Long (Short) 1 close to ATM Call (strike 3)
        4. Short (Long) 1 OTM Call (strike 4)        

        strikes: a dictionary of strikes for the 4 options (strike1 to strike4)
        premiums: a dictionary of premiums for the 4 options (premium1 to premium4)
        expiry: expiration date of the options
        """
        self.premiums = premiums
        self.direction = direction
        self.short_put_option = Option(strikes['strike1'], premiums['premium1'], expiry)
        self.long_put_option = Option(strikes['strike2'], premiums['premium2'], expiry)
        self.long_call_option = Option(strikes['strike3'], premiums['premium3'], expiry)
        self.short_call_option = Option(strikes['strike4'], premiums['premium4'], expiry)

    def initial_cashflow(self):
        """
        Calculate the initial cashflow of the Iron Condor strategy.
        """
        if self.direction == "long":
            initial_cashflow = self.premiums['premium1'] - self.premiums['premium2'] - self.premiums['premium3'] + self.premiums['premium4']
        elif self.direction == "short":
            initial_cashflow = -self.premiums['premium1'] + self.premiums['premium2'] + self.premiums['premium3'] - self.premiums['premium4']
        
        return initial_cashflow

    def insert_spot(self, spot_price: float):
        """
        Insert the current spot price for all the options in the Iron Condor strategy.
        """
        self.short_put_option.insert_spot(spot_price)
        self.long_put_option.insert_spot(spot_price)
        self.long_call_option.insert_spot(spot_price)
        self.short_call_option.insert_spot(spot_price)
    
    def payoff(self):
        """
        Calculate the total payoff (profit or loss) of the Iron Condor strategy.
        
        direction: 'long' for long Iron Condor, 'short' for short Iron Condor
        """
        if self.direction == "long":
            short_put = self.short_put_option.short_put()
            long_put = self.long_put_option.long_put()
            long_call = self.long_call_option.long_call()
            short_call = self.short_call_option.short_call()
            total_payoff = short_put + long_put + long_call + short_call
        elif self.direction == "short":
            long_put = self.short_put_option.long_put()
            short_put = self.long_put_option.short_put()
            short_call = self.long_call_option.short_call()
            long_call = self.short_call_option.long_call()
            total_payoff = long_put + short_put + short_call + long_call
        
        return total_payoff

class OptionsBacktester:
    def __init__(self,
                 signal_df: pd.DataFrame,
                 spot_price_df: pd.DataFrame,
                 options_df: pd.DataFrame,
                 upper_threshold: float,
                 lower_threshold: float,
                 option_type: str,
                 dynamic_option_strategy: bool = False,
                 dynamic_strike: bool = False,):
        self.signal_df = signal_df
        self.signal_df.columns = ["signal"]
        self.spot_price_df = spot_price_df
        self.options_df = options_df
        self.upper_threshold = upper_threshold
        self.lower_threshold = lower_threshold
        self.option_type = option_type
        self.dynamic_option_strategy = dynamic_option_strategy
        self.dynamic_strike = dynamic_strike

        self.comb_df = pd.concat([self.signal_df, self.spot_price_df.shift(), self.options_df.shift()], axis=1)
        self.comb_df["action"] = np.where(self.comb_df["signal"] > self.upper_threshold, 1, np.where(self.comb_df["signal"] < self.lower_threshold, -1, 0))

    def long_short_vol_df(self):
        self.trading_df = {}
        for index, row in self.comb_df.iterrows():
            if row["action"] != 0:
                self.trading_df[index] = self.comb_df.loc[index]

## Code to filter down the options data to only rows where strike matches spot

In [142]:
upper_threshold = 1
lower_threshold = -1

aapl_vrp = pd.read_csv("data/AAPL_vrp_standardised.csv", parse_dates=True, index_col=0)

aapl = pd.read_hdf("data/all_tickers_time_series.hf5", key="AAPL").drop_duplicates().set_index("date")

aapl.index = pd.to_datetime(aapl.index)

options_data = pd.read_hdf("data/all_options_data.h5", key="AAPL").set_index("date")
options_data['exdate'] = pd.to_datetime(options_data['exdate'])
options_data.index = pd.to_datetime(options_data.index)


# options_data_filtered = pd.DataFrame()
options_data['price'] = aapl["prc"]
options_data["strike_price"] = options_data["strike_price"] / 1000
# for date in options_data.index.unique():
#     if options_data.loc[date]["exdate"].nunique() > 1:
#         options_data_filtered = pd.concat([options_data_filtered, options_data.loc[date].query("exdate == exdate.min()")], axis=0)


options_data_filtered = options_data.groupby("date").apply(lambda x: x.query("exdate == exdate.min()")).reset_index(level=0, drop=True)

# options_data_filtered["strike_minus_price"] = np.abs(options_data_filtered["strike_price"] - options_data_filtered["price"])
# new_df = options_data_filtered.groupby("date").apply(lambda x: x.query("strike_minus_price == strike_minus_price.min()")).reset_index(level=0, drop=True)
# date_counts = new_df.index.value_counts()
# def filter_max_open_interest(x):
#     if len(x) >= 2:
#         return x.query("open_interest == open_interest.max()")
#     return x 

# new_new_df = new_df.groupby(['date', 'cp_flag']).apply(filter_max_open_interest).reset_index(level=[0, 1], drop=True)

# comb_df = pd.concat([new_new_df, aapl_vrp], axis=1) # the above block is only used if i want to filter the options data for strike close to spot
comb_df = pd.concat([options_data_filtered, aapl_vrp], axis=1)
comb_df = comb_df.dropna()
comb_df["action"] = np.where(comb_df["vrp_standardised"] > upper_threshold, 1, np.where(comb_df["vrp_standardised"] < lower_threshold, -1, 0))

In [144]:
comb_df

,ticker,exdate,cp_flag,strike_price,best_bid,best_offer,open_interest,impl_volatility,delta,gamma,theta,vega,volume,price,vrp_standardised,action
2020-06-01,AAPL,2020-06-05,C,287.5,34.35,34.75,61.0,0.566173,0.973477,0.003222,-53.77634,2.071664,6.0,321.85001,-0.707107,0
2020-06-01,AAPL,2020-06-05,C,290.0,31.95,32.15,498.0,0.529447,0.971835,0.003623,-52.87457,2.179710,32.0,321.85001,-0.707107,0
2020-06-01,AAPL,2020-06-05,C,292.5,29.45,29.75,129.0,0.513018,0.964684,0.004507,-61.71273,2.622997,6.0,321.85001,-0.707107,0
2020-06-01,AAPL,2020-06-05,C,295.0,27.00,27.20,2438.0,0.475118,0.962196,0.005145,-60.43238,2.773249,1450.0,321.85001,-0.707107,0
2020-06-01,AAPL,2020-06-05,C,297.5,24.50,24.80,183.0,0.453182,0.953750,0.006350,-67.82130,3.268199,17.0,321.85001,-0.707107,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-31,AAPL,2023-09-01,P,177.5,0.01,0.02,14181.0,0.466922,-0.009593,0.005606,-21.47833,0.253209,3744.0,187.87000,1.165067,1
2023-08-31,AAPL,2023-09-01,P,180.0,0.02,0.03,23962.0,0.392160,-0.017771,0.011368,-30.68410,0.431584,13432.0,187.87000,1.165067,1
2023-08-31,AAPL,2023-09-01,P,182.5,0.04,0.05,20410.0,0.312088,-0.036579,0.026129,-44.56422,0.787846,9587.0,187.87000,1.165067,1
2023-08-31,AAPL,2023-09-01,P,185.0,0.12,0.13,21667.0,0.241647,-0.108708,0.078512,-79.87569,1.832929,34341.0,187.87000,1.165067,1


In [38]:
xlu_options = pd.read_hdf("data/etf_options_data.hf5", key="XLU")
xlu_options = xlu_options.set_index("date")
xlu_options["strike_price"] = xlu_options["strike_price"] / 1000
xlu_filtered_options = xlu_options.groupby("date").apply(lambda x: x.query("exdate == exdate.min()")).reset_index(level=0, drop=True)

In [52]:
xlu_filtered_options.loc["2020-01-02"][xlu_filtered_options.loc["2020-01-02"]['cp_flag'] == 'P']

,ticker,exdate,cp_flag,strike_price,best_bid,best_offer,open_interest,impl_volatility,delta,gamma,theta,vega,volume
date,,,,,,,,,,,,,
2020-01-02,XLU,2020-01-03,P,70.5,6.65,6.75,0.0,0.838565,-0.988134,0.011432,-15.239890,0.103290,0.0
2020-01-02,XLU,2020-01-03,P,71.5,7.65,7.80,0.0,1.109253,-0.973461,0.016762,-40.860840,0.205001,0.0
2020-01-02,XLU,2020-01-03,P,55.5,0.00,0.02,0.0,1.119412,-0.007928,0.005825,-14.851920,0.072606,0.0
2020-01-02,XLU,2020-01-03,P,69.5,5.65,5.75,0.0,0.734057,-0.986642,0.014467,-14.761690,0.114286,0.0
2020-01-02,XLU,2020-01-03,P,69.0,5.15,5.25,0.0,0.680518,-0.985710,0.016540,-14.494670,0.120995,0.0
2020-01-02,XLU,2020-01-03,P,68.0,4.15,4.25,0.0,0.570450,-0.983279,0.022581,-13.878340,0.138519,0.0
2020-01-02,XLU,2020-01-03,P,67.5,3.65,3.75,0.0,0.513708,-0.981645,0.027155,-13.517800,0.150241,0.0
2020-01-02,XLU,2020-01-03,P,67.0,3.15,3.25,0.0,0.455635,-0.979580,0.033524,-13.107730,0.164566,0.0
2020-01-02,XLU,2020-01-03,P,66.5,2.68,2.75,0.0,0.457663,-0.956911,0.060735,-24.868520,0.305536,0.0


In [53]:
xlu_time_series = pd.read_hdf("data/etf_series.hf5", key="XLU")
xlu_time_series.loc[["2020-01-03"]]["Close"]

Date
2020-01-03    63.939999
Name: Close, dtype: float64

In [58]:
strikes = {
    'strike1': 63,  # Short put
    'strike2': 64.5, # Long put
    'strike3': 64.5, # Long call
    'strike4': 66  # Short call
}

premiums = {
    'premium1': 0.01,  # Short put premium
    'premium2': 0.75,  # Long put premium
    'premium3': 0.01,  # Long call premium
    'premium4': 0   # Short call premium
}

expiry = "2024-12-31"

ironcondor = IronCondor(direction="long", strikes=strikes, premiums=premiums, expiry=expiry)
print(ironcondor.initial_cashflow())
ironcondor.insert_spot(63.94)
ironcondor.payoff()

-0.75


-0.18999999999999773